# 05 — Privilege Escalation через `SECURITY DEFINER`

> **`vuln_class`:** `PRIV_ESCALATE` · **Риск:** 8/10 · **CWE-269** · **CAPEC-470**

Функция в PostgreSQL объявлена с атрибутом `SECURITY DEFINER` — она выполняется с правами **владельца функции**, а не вызывающего. Если у функции **не зафиксирован `search_path`**, атакующий может подсунуть свою таблицу/функцию в `pg_temp` и **выполнить код в роли владельца** (часто это `postgres` — суперюзер).


## 🧒 Аналогия для ребёнка

Представь, что папа разрешил тебе **с его карточки** покупать
молоко в магазине у дома. Он подписал инструкцию: «купить молоко».
Карточка работает только когда ты несёшь молоко.

Хитрый братик переклеивает в магазине ценники: на пачке жвачки
пишет «молоко». Ты приходишь, кладёшь жвачку, кассир видит надпись
«молоко», списывает с папиной карты. Хотя ты унёс жвачку, а не молоко.

В SQL: функция `SECURITY DEFINER` ходит в БД от имени владельца.
Если внутри функции написано просто «возьми из таблицы `users`»,
а атакующий ДО вызова **создал свою таблицу `users` в `pg_temp`** —
функция возьмёт **подделку атакующего** (потому что `pg_temp`
выше в `search_path`).


## ⚠️ Дисклеймер

SQLite не имеет `SECURITY DEFINER`, `SET search_path`, `pg_temp`.
Эти концепции — **строго про PostgreSQL**. Здесь мы **симулируем**
атаку через Python-обёртку: имитируем «search_path» как простой
словарь, видим как подмена работает.

Если хочешь воспроизвести атаку **на настоящем Postgres**, см. PG docs:
https://www.postgresql.org/docs/current/sql-createfunction.html#SQL-CREATEFUNCTION-SECURITY


## 1. Setup — имитация Postgres + SECURITY DEFINER функции


In [ ]:
"""
@brief Подготовка окружения и mock-БД через in-memory SQLite.
@details
    Никаких внешних зависимостей кроме stdlib + sqlite3 (есть в Colab из коробки).
    SQLite используем как «упрощённую модель PostgreSQL» — он умеет
    почти весь стандартный SQL, что достаточно для демонстраций уязвимостей.
@note
    Реальная система работает на PostgreSQL (см. ADR-0001),
    использует pglast для AST-парсинга. Здесь, для наглядности,
    эмулируем аудитор через `re` (регулярки) и простой pattern matching.
"""
import sqlite3
import re
import time
from textwrap import dedent


def section(title):
    """@brief Печатает заголовок секции."""
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def show_result(rows, max_rows=10):
    """@brief Печатает результаты запроса в виде таблицы."""
    if not rows:
        print("  (нет строк)")
        return
    for i, r in enumerate(rows[:max_rows]):
        print(f"  {i + 1:>3}. {r}")
    if len(rows) > max_rows:
        print(f"  ... ещё {len(rows) - max_rows} строк")


def print_finding(f):
    """@brief Красиво печатает Finding от нашего аудитора."""
    print(f"  ⚠️  {f['rule_id']}")
    print(f"      vuln_class:  {f['vuln_class']}")
    print(f"      severity:    {f['severity']}")
    print(f"      risk_score:  {f['risk_score']}/10")
    print(f"      message:     {f['message']}")
    if f.get("evidence_refs"):
        print(f"      ссылки:      {', '.join(f['evidence_refs'])}")


##
# @brief Имитация PostgreSQL search_path.
# @details
#   В Postgres `search_path` — список схем, по которым ищется неквалифицированное
#   имя объекта. Дефолт: 'pg_temp, "$user", public'. То есть pg_temp ВПЕРЕДИ public.
#   Если атакующий создаст в pg_temp функцию users(), она «затмит» public.users.
SEARCH_PATH = ["pg_temp", "public"]

# Имитация двух схем
SCHEMAS = {
    "public": {
        "users": [
            (1, "alice", "admin"),
            (2, "bob",   "user"),
        ],
    },
    "pg_temp": {},  # сюда атакующий может «положить» свою таблицу
}


def resolve_table(name):
    """@brief Имитация PG: ищет таблицу по search_path."""
    for schema in SEARCH_PATH:
        if name in SCHEMAS[schema]:
            return schema, SCHEMAS[schema][name]
    raise KeyError(f"таблица {name} не найдена")


print("Имитация PG: SEARCH_PATH =", SEARCH_PATH)
print("Содержимое public.users:")
show_result(SCHEMAS["public"]["users"])


## 2. Уязвимая `SECURITY DEFINER` функция — без `SET search_path`


In [ ]:
##
# @brief УЯЗВИМАЯ функция: SECURITY DEFINER без SET search_path.
# @details
#   Имитируем поведение PG:
#     CREATE FUNCTION admin_lookup(login text) RETURNS bigint
#       LANGUAGE plpgsql SECURITY DEFINER  -- ⚠️ нет SET search_path
#     AS $$ SELECT id FROM users WHERE login = $1 $$;
# @warning  Берёт неквалифицированное имя `users` — может попасть в pg_temp.
def admin_lookup_BAD(login):
    schema, table = resolve_table("users")
    print(f"  Функция читает users из схемы '{schema}' (search_path выбор)")
    for row in table:
        if row[1] == login:
            return row
    return None


section("Нормальный вызов: ищем alice")
print("  Результат:", admin_lookup_BAD("alice"))


## 3. Атака: search_path hijacking


In [ ]:
section("АТАКА: атакующий кладёт свою таблицу users в pg_temp")
SCHEMAS["pg_temp"]["users"] = [
    (999, "alice", "admin"),     # имя как в public, но id=999 — поддельный
    (998, "fake_admin", "admin"),
]

section("Тот же вызов admin_lookup_BAD('alice')")
result = admin_lookup_BAD("alice")
print(f"  💀 Результат: {result}")
print(f"  💀 Функция вернула ПОДДЕЛЬНЫЕ данные из pg_temp,")
print(f"  💀 потому что pg_temp выше public в search_path.")
print(f"  💀 Если функция дальше что-то делает с этим id — выполнится с правами владельца.")


## 4. Аудитор Phase 1 — правило R007

Phase 1 в проде смотрит AST `CreateFunctionStmt` и проверяет:
- есть ли атрибут `security definer`;
- есть ли `SET search_path`.

Здесь — regex по DDL-тексту.


In [ ]:
##
# @brief Phase 1 R007 — детект SECURITY DEFINER без SET search_path.
def audit_R007_security_definer(ddl_text):
    findings = []
    if not re.search(r"\bSECURITY\s+DEFINER\b", ddl_text, re.IGNORECASE):
        return findings
    if not re.search(r"\bSET\s+search_path\s*=", ddl_text, re.IGNORECASE):
        findings.append({
            "rule_id":       "R007-security-definer-no-search-path",
            "vuln_class":    "PRIV_ESCALATE",
            "severity":      "high", "risk_score": 8,
            "message":       "SECURITY DEFINER без SET search_path — search_path hijack",
            "evidence_refs": ["CWE-269", "CAPEC-470", "PG-docs#sql-createfunction"],
        })
    return findings


bad_ddl = """
CREATE OR REPLACE FUNCTION admin_lookup(login text) RETURNS bigint
LANGUAGE plpgsql
SECURITY DEFINER
AS $$ DECLARE r bigint; BEGIN SELECT id INTO r FROM users WHERE login=$1; RETURN r; END $$;
"""
good_ddl = """
CREATE OR REPLACE FUNCTION admin_lookup(login text) RETURNS bigint
LANGUAGE plpgsql
SECURITY DEFINER
SET search_path = pg_catalog, pg_temp
AS $$ DECLARE r bigint; BEGIN SELECT id INTO r FROM public.users WHERE login=$1; RETURN r; END $$;
"""

section("Аудитор по плохой версии")
for f in audit_R007_security_definer(bad_ddl):
    print_finding(f)

section("Аудитор по хорошей версии")
fs = audit_R007_security_definer(good_ddl)
if fs:
    for f in fs:
        print_finding(f)
else:
    print("  ✅ Уязвимостей не найдено.")


## 5. Безопасная функция


In [ ]:
section("Безопасная DDL (только текст, мы не исполняем — SQLite не PG)")
print(good_ddl)

print()
print("Ключевые отличия:")
print("  1. SET search_path = pg_catalog, pg_temp — фиксируем порядок схем.")
print("  2. FROM public.users — квалифицированное имя, не оставляем resolver-у выбор.")
print("  3. (отдельно) REVOKE ALL ON FUNCTION ... FROM PUBLIC; GRANT EXECUTE TO app_role.")


## Итог

Мы увидели одно и то же на двух функциях:

- **Уязвимая** — украли данные / повредили БД / поднялись в правах.
- **Безопасная** — та же атака уходит в пустоту.

Между ними — **один аудитор** с конкретным правилом, которое можно
запустить детерминированно (без LLM) на каждом сгенерированном SQL.

## Куда дальше

- **Описание уязвимости (под микроскопом):** [problems/vulnerabilities/05-privilege-escalation-execute/README.md](../problems/vulnerabilities/05-privilege-escalation-execute/README.md)
- **Варианты решения + почему так:** [problems/vulnerabilities/05-privilege-escalation-execute/solutions.md](../problems/vulnerabilities/05-privilege-escalation-execute/solutions.md)
- **Архитектура цикла:** [docs/adr/0002-loop-architecture-langgraph.md](../docs/adr/0002-loop-architecture-langgraph.md)
- **Гибридный аудитор (pglast + LLM):** [docs/adr/0004-hybrid-auditor-ast-plus-llm.md](../docs/adr/0004-hybrid-auditor-ast-plus-llm.md)
